# Scanpy Extra: (For quick testing)

In [37]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1, 0.2, 0.3] 

# Check Vln clusters

In [ ]:
# L1 clusters
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)

logger.info("Loading L1 data ...")
adata = sc.read(scanpy_dir + 'adata_clusters.h5ad')

cluster_anns = {
    '0': 'ExN-UL',
    '1': 'ExN-DL',
    '2': 'RG',
    '3': 'InN',
    '4': 'Endo-Peri',
    '5': 'OPC',
    '6': 'MG'
}

genes = [
    "CUX2", "SATB2", "PLXNA4", "DSCAML1",         
    "TLE4", "SULF2", "LMO3",                       
    "GAD1", "GAD2", "ERBB4",  "BCL11B",            
    "RELN", "TP73",             
    "GLI3", "GLI2", "PRDM16", "PAX6", "SOX2",      
    "COL4A1", "FN1",                              
    "PDGFRA",                                      # Oligodendrocyte progenitor cells         
    "C3",                                          
]

final_genes = [
    (
        "Final marker genes",
        [
            "CUX2", "SATB2", "PLXNA4", "DSCAML1",      # UL Excitatory cortical neurons
            "TLE4", "SULF2", "LMO3",                   # DL Excitatory cortical neurons
            "GAD1", "GAD2", "ERBB4", "BCL11B",         # GABAergic inhibitory neurons
            "RELN", "TP73", "SPON1", "MIK67",          # Extra genes
            "GLI3", "GLI2", "PRDM16", "PAX6", "SOX2",  # Radial glia
            "COL4A1", "FN1",                           # Endothelial/vascular cells
            "PDGFRA",                                  # Oligodendrocyte progenitor cells
            "C3",                                      # Microglia
        ]
    )
]

obs_columns = ['leiden_harmony_0.1']

logger.info("Plot L1 Vln ...")
fig = plot_filtered_violin(
    adata, 
    final_genes, 
    groupby_base="leiden_harmony", 
    resolutions=[0.1], 
    clustering_algorithm="Leiden (harmony)")
plt.show()

logger.info("Plot L1 UMAP ...")
plot_umap_grid(adata, obs_columns, grid_size=(1, 1), figsize=(16, 8), save_path=None)

# Filter to genes present in the dataset 
genes_present = [
    g for g in genes
    if g in adata.var_names or g in getattr(adata.raw, "var_names", [])
]

ncols = 4
nrows = math.ceil(len(genes_present) / ncols)

logger.info("Plot L1 Feature plots ...")
sc.pl.umap(
    adata,
    color=genes_present,
    ncols=ncols,
    vmin=0,
    vmax="p99",     # robust scaling for sparse scRNA-seq
    cmap="viridis",
    size=5,
    frameon=False,
    show=True,
)

# Free memory
del adata
gc.collect()

# L2 clusters
logger.info("Processing L2 subclusters ...")
subcluster_dir = scanpy_dir + 'subclustering/'

cell_types = ['ExN-UL', 'ExN-DL', 'RG', 'InN']

resolution_map = {
    'ExN-UL': 0.2,
    'ExN-DL': 0.2,
    'RG': 0.2,
    'InN': 0.1
}

for cell_type in cell_types:

    logger.info(f"Loading L2 data for {cell_type} ...")
    adata_sub = sc.read(
        subcluster_dir + f'adata_{cell_type}_subclusters.h5ad'
    )

    logger.info(f"Plot L2 Vln for {cell_type} ...") 
    valid_genes = [g for g in final_genes if g in adata_sub.var_names]
    sc.pl.stacked_violin(
        adata_sub,
        var_names=valid_genes,
        groupby='subcluster',
        swap_axes=True,
        show=True
    )
    
    logger.info(f"Plot L2 UMAP for {cell_type} ...")  
    sc.pl.umap(
        adata_sub,
        color='subcluster',
        title=f'{cell_type} subcluster: Harmony L2',
        show=True
    )

    # Filter to genes present in the dataset 

    if valid_genes:
            ncols = 4

            sc.pl.umap(
                adata_sub,
                color=valid_genes,
                ncols=ncols,
                vmin=0,
                vmax="p99",
                cmap="viridis",
                size=5,
                frameon=False,
                show=True,
            )
    else:
        logger.warning(f"No valid genes found for {cell_type}; skipping feature plots.")

    del adata_sub
    gc.collect()

logger.info("All done.")